# Solutions — HTTP with requests: GET, POST, Status Codes, Clients

Exercise notebook: `07_apis/01_http_requests_basics.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

### Setup (same as the exercise notebook)

In [ ]:
import sys
from pathlib import Path
LAB = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets").is_dir() and (p / "labkit").is_dir())   # lab root (works from any folder)
sys.path.insert(0, str(LAB))                                # make `labkit` importable from the notebook
from labkit.mock_api import MockSession, BASE_URL
import requests

session = MockSession()                                     # offline fake with the same surface as requests.Session
resp = session.get(f"{BASE_URL}/customers", params={"page": 1, "page_size": 3}, timeout=5)
print(resp.status_code, resp.json()["total"], [c["customer_id"] for c in resp.json()["data"]])
# Prefer real HTTP? In a terminal:  python -m labkit.mock_server --port 8765
# then use requests.Session() with base_url "http://127.0.0.1:8765/v1" — your functions shouldn't change at all.

## Exercise 1 — 🟢 Easy · GET a resource, 404 → None

**Working solution**

In [ ]:
def get_customer(session, base_url, customer_id):
    resp = session.get(f"{base_url}/customers/{customer_id}", timeout=10)
    if resp.status_code == 404:
        return None
    resp.raise_for_status()
    return resp.json()

**Explanation**

404 for a lookup is often an expected outcome, not an error; everything else non-2xx should fail loudly. `raise_for_status()` raises `requests.HTTPError` carrying `.response`.

**Complexity:** O(1) requests.

**Common mistakes**

- No timeout → a hung server hangs your job forever.
- Checking `resp.text` instead of the status code.

## Exercise 2 — 🟡 Medium · POST JSON and handle validation errors

**Working solution**

In [ ]:
def create_order(session, base_url, customer_id, items):
    resp = session.post(f"{base_url}/orders", json={"customer_id": customer_id, "items": items}, timeout=10)
    if resp.status_code == 422:
        raise ValueError("; ".join(resp.json().get("errors", ["validation failed"])))
    resp.raise_for_status()
    return resp.json()["order_id"]

**Explanation**

`json=` serializes the body and sets `Content-Type: application/json` (don't `json.dumps` it yourself into `data=`).

**Complexity:** O(1).

**Common mistakes**

- `data=json.dumps(body)` without the header.
- Assuming 200 — creation endpoints return 201.

## Exercise 3 — 🟡 Medium · Build URLs with query parameters safely

**Working solution**

In [ ]:
from urllib.parse import urlencode

def build_url(base, path, **params):
    url = f"{base.rstrip('/')}/{path.lstrip('/')}"
    clean = {}
    for key, value in params.items():
        if value is None:
            continue
        if isinstance(value, bool):
            value = "true" if value else "false"
        clean[key] = value
    query = urlencode(clean, doseq=True)
    return f"{url}?{query}" if query else url

**Explanation**

`urlencode` escapes spaces, `&`, `=` and unicode correctly; `doseq=True` expands sequences. (With requests you'd usually pass `params=` and let it do this.)

**Complexity:** O(params).

**Common mistakes**

- f-string query building → `name=Kalyan B&x` breaks the URL.
- Sending `True` as `'True'` — most APIs expect lowercase.

## Exercise 4 — 🟠 Hard · A typed API client with an exception hierarchy

**Working solution**

In [ ]:
class ApiError(Exception):
    def __init__(self, message, status=None, payload=None):
        super().__init__(message)
        self.status = status
        self.payload = payload

class AuthError(ApiError): pass
class NotFound(ApiError): pass
class ServerError(ApiError): pass

class RateLimited(ApiError):
    def __init__(self, message, status=429, payload=None, retry_after=None):
        super().__init__(message, status, payload)
        self.retry_after = retry_after


class ApiClient:
    def __init__(self, session, base_url, api_key=None, timeout=5):
        self.session = session
        self.base_url = base_url.rstrip("/")
        self.timeout = timeout
        self.headers = {"Accept": "application/json"}
        if api_key:
            self.headers["Authorization"] = f"Bearer {api_key}"

    def _handle(self, resp):
        if resp.ok:
            return resp.json()
        try:
            payload = resp.json()
        except ValueError:
            payload = None
        status = resp.status_code
        message = f"HTTP {status} for {getattr(resp, 'url', '')}"
        if status in (401, 403):
            raise AuthError(message, status, payload)
        if status == 404:
            raise NotFound(message, status, payload)
        if status == 429:
            raw = resp.headers.get("Retry-After")
            retry_after = float(raw) if raw and raw.replace(".", "", 1).isdigit() else None
            raise RateLimited(message, status, payload, retry_after)
        if status >= 500:
            raise ServerError(message, status, payload)
        raise ApiError(message, status, payload)

    def get_json(self, path, **params):
        resp = self.session.get(f"{self.base_url}/{path.lstrip('/')}", params=params or None,
                                headers=self.headers, timeout=self.timeout)
        return self._handle(resp)

    def post_json(self, path, body):
        resp = self.session.post(f"{self.base_url}/{path.lstrip('/')}", json=body, headers=self.headers, timeout=self.timeout)
        return self._handle(resp)

**Explanation**

A small exception hierarchy lets callers choose granularity: `except RateLimited` for backoff, `except ApiError` for "anything from the API".
Subclasses with `pass` bodies are normal in Python — the class *is* the information.

**Complexity:** O(1).

**Common mistakes**

- Returning `(status, body)` tuples and making every caller re-check status codes.
- Wrapping timeouts into ApiError and losing the retryable signal.

## Challenge — 🔴 Very Hard · Concurrent lookups with a thread pool

**Working solution**

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def _fetch_one(session, base_url, cid):
    resp = session.get(f"{base_url}/customers/{cid}", timeout=10)
    if resp.status_code == 404:
        return None
    resp.raise_for_status()
    return resp.json()

def fetch_many(session, base_url, ids, max_workers=8):
    found, missing, failed = {}, [], {}
    unique = list(dict.fromkeys(ids))
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        futures = {pool.submit(_fetch_one, session, base_url, cid): cid for cid in unique}
        for future in as_completed(futures):
            cid = futures[future]
            try:
                record = future.result()
            except Exception as exc:
                failed[cid] = f"{type(exc).__name__}: {exc}"
                continue
            if record is None:
                missing.append(cid)
            else:
                found[cid] = record
    return found, sorted(missing), failed

**Explanation**

Network calls spend their time waiting, and the GIL is released while waiting — so threads give real speed-ups for I/O. A dict
`{future: id}` maps results back to inputs; `future.result()` re-raises the worker's exception in the caller, where you can classify it.
(`requests.Session` is generally safe to share for simple GETs; for heavy use give each thread its own session.)

**Complexity:** ≈ O(n / workers) wall-clock for I/O-bound calls.

**Common mistakes**

- Using processes for I/O-bound work (overhead, pickling).
- Letting one exception propagate out of the `with` block and lose the other results.